# Engineering LLM Systems - Tokenization & State Management

Welcome back, architects and engineers. Today we transition from "using LLMs" to "engineering with LLMs." 

We will explore two critical architectural constraints:
1. **Sub-word Tokenization**: Analyzing how structured data (JSON, code, logs) consumes your context window.
2. **The Statelessness Constraint**: Engineering sophisticated conversation state in distributed systems.

Let's get started.

## Section 1: Tokenization of Structured Data

In production, you aren't just sending "Hello world." You're sending system prompts, RAG results, and API schemas. 

**The Architectural Impact**: Whitespace, brackets, and specific syntax in code/JSON can inflate token counts unexpectedly. Understanding this is key to optimizing latency and cost.

In [3]:
import tiktoken

# Load Encoing model
encoding = tiktoken.encoding_for_model("gpt-4o")

# Encode a String
tokens = encoding.encode("Hello, world! Hello, world! Hello, world! Hello, world! Hello, world!")
print(f"Total tokens: {len(tokens)}")


Total tokens: 20


In [14]:
# Load Encoing model
encoding = tiktoken.encoding_for_model("gpt-4o")

#A structured system configuration snippet
technical_payload = """
{
  "deployment": "k8s-cluster-01",
  "resources": {
    "cpu": "500m",
    "memory": "2Gi"
  },
  "scaling": {"min": 2, "max": 10}
}
""".strip()

tokens = encoding.encode(technical_payload)
print(f"Total tokens: {len(tokens)}")

print(f"{'ID':<8} | {'Representation'}")
print("-" * 25)
for token_id in tokens: # Inspecting the first 20 tokens
    token_text = encoding.decode([token_id])
    print(f"{token_id:<8} | '{token_text}'")


Total tokens: 55
ID       | Representation
-------------------------
745      | '{
'
220      | ' '
392      | ' "'
195280   | 'deployment'
1243     | '":'
392      | ' "'
74       | 'k'
23       | '8'
82       | 's'
29640    | '-cl'
7282     | 'uster'
12       | '-'
2290     | '01'
1150     | '",
'
220      | ' '
392      | ' "'
25623    | 'resources'
1243     | '":'
405      | ' {
'
271      | '   '
392      | ' "'
38581    | 'cpu'
1243     | '":'
392      | ' "'
3234     | '500'
76       | 'm'
1150     | '",
'
271      | '   '
392      | ' "'
37835    | 'memory'
1243     | '":'
392      | ' "'
17       | '2'
49130    | 'Gi'
1092     | '"
'
220      | ' '
1862     | ' },
'
220      | ' '
392      | ' "'
2786     | 'sc'
7330     | 'aling'
1243     | '":'
10494    | ' {"'
1493     | 'min'
1243     | '":'
220      | ' '
17       | '2'
11       | ','
392      | ' "'
3228     | 'max'
1243     | '":'
220      | ' '
702      | '10'
739      | '}
'
92       | '}'


## Section 2: Implementing State in Stateless Architectures

LLM APIs are RESTful and stateless. For a complex troubleshooting agent or a multi-step design assistant, *you* must manage the session state.

In [15]:
import os
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)
client = OpenAI() # Assumes OPENAI_API_KEY is in .env
print("Environment initialized for Architectural Review demo.")

Environment initialized for Architectural Review demo.


### Step 1: Defining the System Context
We provide a complex description of a distributed system.

In [16]:
system_description = """
Our architecture uses a Node.js API Gateway, three Python microservices for data processing, 
and a PostgreSQL primary database with two read-replicas. We use RabbitMQ for asynchronous 
event processing between the services.
"""

messages = [
    {"role": "system", "content": "You are a Senior Cloud Architect."},
    {"role": "user", "content": f"Please acknowledge this system architecture: {system_description}"}
]

response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
print(f"Architect AI: {response.choices[0].message.content}")

Architect AI: Your system architecture sounds robust and well-structured for handling a variety of workloads. Here’s a brief analysis of each component:

1. **Node.js API Gateway**: Using Node.js allows for handling many concurrent connections efficiently, making it suitable for an API gateway. It can serve as a single entry point for clients, routing requests to the appropriate microservices.

2. **Python Microservices**: Python is a great choice for data processing tasks, given its richness in libraries and frameworks. The separation into three microservices allows for modularity, making the services independently deployable and scalable.

3. **PostgreSQL with read-replicas**: This setup provides a good balance between high availability and read scalability. PostgreSQL is widely used for its robustness and feature set, while read replicas help distribute read load and enhance performance.

4. **RabbitMQ for Asynchronous Communication**: RabbitMQ is a reliable message broker that faci

### Step 2: The Stateless Failure
Now we ask a highly contextual question without providing the previous context.

In [17]:
# Attempting to ask about the replicas without restating the architecture
messages = [
    {"role": "system", "content": "You are a Senior Cloud Architect."},
    {"role": "user", "content": "How should we handle the failover for the read-replicas?"}
]

response = client.chat.completions.create(model="gpt-4o-mini", messages=messages)
print(f"Architect AI: {response.choices[0].message.content}")

Architect AI: Handling failover for read replicas is critical for maintaining high availability and performance in a cloud architecture. Here are the steps and best practices to manage failover for read replicas:

### 1. **Determine Your Strategy**
   - **Automatic Failover**: Many cloud providers offer built-in automation for failover; explore tools and services like Amazon RDS Multi-AZ, Google Cloud Spanner, Aurora, or Azure SQL Database’s geo-replication.
   - **Manual Failover**: If automatic options are not available, consider a manual failover process that you can execute in case of failure.

### 2. **Monitoring and Alerts**
   - Set up monitoring for the health of your replicas using tools like AWS CloudWatch, Azure Monitor, or third-party monitoring solutions.
   - Implement alerts that notify your team when a read replica becomes unhealthy or when latency increases significantly.

### 3. **Health Checks**
   - Perform regular health checks on your read replicas to ensure they 

### The Engineering Solution: Message Threading

To solve this, we must maintain a `history` array. In a production environment, this would likely be stored in a Redis cache or a database indexed by a `SessionID`.

Here is how we properly thread the conversation:

In [18]:
history = [
    {"role": "system", "content": "You are a Senior Cloud Architect."},
    {"role": "user", "content": f"Architecture: {system_description}"},
    {"role": "assistant", "content": "I have reviewed the Node.js/Python/PostgreSQL stack with RabbitMQ. How can I help?"},
    {"role": "user", "content": "How should we handle the failover for the read-replicas?"}
]

response = client.chat.completions.create(model="gpt-4o-mini", messages=history)
print(f"Architect AI: {response.choices[0].message.content}")

Architect AI: Handling failover for read-replicas in a PostgreSQL architecture is crucial for ensuring high availability and minimizing downtime. Here are some strategies to consider:

### 1. Monitoring and Health Checks
- **Health Monitoring**: Use a monitoring solution (such as Prometheus, Grafana, or any other tools) to regularly check the health of your primary and read-replica instances.
- **Automated Alerts**: Set up alerts to notify your team immediately if a read-replica fails or becomes unreachable.

### 2. Automatic Failover
- **PgBouncer or HAProxy**: Utilize a connection pooler (like PgBouncer) or a reverse proxy (like HAProxy) to manage connections to your PostgreSQL instances. These tools can handle automatic failover by changing the routing of connections if a read-replica fails.
- **Patroni**: Consider using a tool like Patroni, which automates the management of high availability for PostgreSQL clusters. With Patroni, you can configure automatic failover for both primar

## Engineering Summary

1. **Token Density**: High-density technical data (code/JSON) requires careful monitoring. Prefer concise formats (YAML/Markdown) for long contexts.
2. **The Session Management Pattern**: You are responsible for state. Whether you use a sliding window (keeping only the last N messages) or summarization (condensing old messages), managing this buffer is the core of LLM engineering.
3. **Cost vs. Context**: Each turn of the conversation becomes more expensive because you are resending the entire history. 

**Challenge**: Think about how you would implement a "sliding window" of context in Python. We'll explore that in the next session!